# Parte 1 · Nem todo exemplo é igualmente difícil

**Avaliação com habilidade latente · Instituto Kunumi**

Três classificadores com acurácias praticamente iguais. O código já está pronto: seu trabalho é observar, levantar hipóteses e interpretar onde e por que cada um erra.

**Execute a célula de preparação primeiro.** Depois siga as tarefas em ordem. O código já está pronto: execute, observe e escreva a sua análise. Mude o que quiser para testar hipóteses; as funções de `utils/handson.py` estão documentadas (`help(funcao)` ou Shift + Tab no Jupyter e no Colab).

In [ ]:
import subprocess
import sys
from pathlib import Path

# No Colab, baixa o repositório do tutorial; localmente, encontra a raiz a partir desta pasta.
if "google.colab" in sys.modules:
    ROOT = Path("/content/latent-ability-ml")
    if not ROOT.exists():
        subprocess.check_call(["git", "clone", "-q", "https://github.com/manuelfjr/latent-ability-ml.git", str(ROOT)])
else:
    ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "utils" / "handson.py").exists())

sys.path.insert(0, str(ROOT))
print("Raiz do projeto:", ROOT)

In [ ]:
import matplotlib as mpl
import matplotlib.pyplot as plt

mpl.rcParams.update({
    "figure.figsize": (7, 4),
    "figure.dpi": 110,
    "figure.facecolor": "#F0F0F0",
    "axes.facecolor": "#F0F0F0",
    "savefig.facecolor": "#F0F0F0",
    "axes.edgecolor": "#B4ADA4",
    "axes.labelcolor": "#1C2127",
    "axes.titlecolor": "#1C2127",
    "axes.titleweight": "bold",
    "axes.titlelocation": "left",
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.color": "#B4ADA4",
    "grid.alpha": 0.35,
    "xtick.color": "#5E5E5E",
    "ytick.color": "#5E5E5E",
    "text.color": "#1C2127",
    "legend.frameon": False,
    "lines.linewidth": 2.6,
    "axes.prop_cycle": mpl.cycler(color=["#F04E44", "#4392D2", "#2FB7CF", "#62CCA3", "#F0D449"]),
})

# Escala sequencial de um só tom (azul da paleta de dados) para mapas de dificuldade.
from matplotlib.colors import LinearSegmentedColormap

if "kunumi_azul" not in mpl.colormaps:
    mpl.colormaps.register(LinearSegmentedColormap.from_list("kunumi_azul", ["#B9D2EA", "#4392D2", "#14304F"]))

## Tarefa 1 · Olhe os dados antes dos modelos (~2 min)

Um problema sintético de classificação binária em duas dimensões, com 300 pontos. Execute e observe.

**Pergunta.** Antes de treinar qualquer modelo: em que região você espera erros, e por quê? Anote a sua hipótese; vamos voltar a ela.

**Para explorar.** Algum ponto parece estar "do lado errado"? Que explicações diferentes poderiam existir para ele estar ali?

In [ ]:
from utils.handson import make_toy_classification_dataset, plot_classification_dataset

df = make_toy_classification_dataset(n_samples=300, class_sep=1.4, random_state=56)

ax = plot_classification_dataset(df)
ax.set_title("Duas classes")
plt.show()

*Sua análise:* 

## Tarefa 2 · Três modelos, quase o mesmo número (~3 min)

Árvore de decisão, k-vizinhos e regressão logística, avaliados nos mesmos 90 exemplos de teste, com três métricas agregadas.

**Pergunta.** Se você tivesse de escolher um modelo só com esta tabela, qual escolheria, e com que confiança? O que mais você gostaria de saber antes de decidir?

**Para explorar.** As três métricas concordam entre si? Em que situação acurácia, acurácia balanceada e F1 contariam histórias diferentes?

In [ ]:
from utils.handson import evaluate_models_on_dataset, summarize_classification_results

# Árvore de decisão, k-vizinhos e regressão logística; 30% dos dados (90 exemplos) ficam para teste
resultados = evaluate_models_on_dataset(df, scenario="ita", random_state=56)

summarize_classification_results(resultados).set_index("model")[["accuracy", "balanced_accuracy", "f1"]].round(3)

*Sua análise:* 

## Tarefa 3 · Quem errou o quê? (~3 min)

O mesmo resultado como matriz de respostas: modelos nas linhas, exemplos de teste nas colunas, `1` para acerto e `0` para erro. Aparecem só as colunas em que algum modelo errou.

**Pergunta.** Que padrões você vê? Os modelos erram nos mesmos exemplos? Algum deles parece ter um "estilo" próprio de erro?

**Para explorar.** Quantas das 90 colunas sobraram? O que as colunas que desapareceram (todos acertam) dizem sobre o teste como instrumento de medida?

In [ ]:
# Modelos nas linhas, exemplos de teste nas colunas, 1 = acerto e 0 = erro
respostas = resultados.pivot(index="model", columns="example_id", values="correct")
print(respostas.shape, "→ (modelos, exemplos de teste)")

# Só as colunas em que pelo menos um modelo errou
respostas.loc[:, respostas.min(axis=0) == 0]

*Sua análise:* 

## Tarefa 4 · Onde moram os exemplos difíceis (~4 min)

Uma primeira medida de dificuldade, sem modelo latente: a fração de modelos que erra cada exemplo (o análogo do índice de dificuldade da teoria clássica dos testes), desenhada no plano das features.

**Pergunta.** Sua hipótese da primeira tarefa se confirmou? Algum exemplo difícil parece mais um problema do rótulo do que do modelo?

**Para explorar.** Que limitação tem essa medida? Ela trata da mesma forma o erro de um modelo forte e o de um fraco?

In [ ]:
dificuldade = 1 - respostas.mean(axis=0)   # fração de modelos que erraram cada exemplo
teste = df.set_index("example_id").loc[respostas.columns].assign(dificuldade=dificuldade)

fig, ax = plt.subplots()
for label, cor in [(0, "#B4ADA4"), (1, "#5E5E5E")]:
    grupo = teste[(teste.label == label) & (teste.dificuldade == 0)]
    ax.scatter(grupo.feature_1, grupo.feature_2, s=16, color=cor, label=f"classe {label}: todos acertam")
erros = teste[teste.dificuldade > 0]
ax.scatter(erros.feature_1, erros.feature_2, s=60 + 200 * erros.dificuldade, color="#F04E44",
           edgecolor="#1C2127", linewidth=0.5, label="algum modelo erra (tamanho = fração de erros)")
ax.set_title("Onde os modelos erram")
ax.legend(fontsize=8)
plt.show()

erros.sort_values("dificuldade", ascending=False)[["feature_1", "feature_2", "label", "dificuldade"]].round(2)

*Sua análise:* 

## Livre: crie o seu cenário (~3 min)

Os mesmos três modelos, agora num cenário que você controla: separação entre classes, ruído de rótulo e desbalanceamento.

**Pergunta.** Que cenário você testou, o que esperava e o que aconteceu? Alguma métrica agregada enganaria você nesse cenário?

**Para explorar.** Algumas ideias: classes muito desbalanceadas (`peso_classe_0 = 0.9`) e compare acurácia com acurácia balanceada; muito ruído de rótulo (`flip_y = 0.15`); classes quase sobrepostas (`class_sep = 0.5`).

In [ ]:
# Mude o que quiser. Antes de executar, anote o que você espera que aconteça.
class_sep = 1.4       # separação entre as classes
flip_y = 0.02         # fração de rótulos trocados ao acaso
peso_classe_0 = 0.5   # 0.5 = classes equilibradas; 0.9 = 90% dos exemplos na classe 0

dados = make_toy_classification_dataset(n_samples=300, class_sep=class_sep, flip_y=flip_y,
                                        weights=(peso_classe_0, 1 - peso_classe_0), random_state=56)
res = evaluate_models_on_dataset(dados, scenario="livre", random_state=56)
tabela = res.pivot(index="model", columns="example_id", values="correct")

print("exemplos que algum modelo erra:", int((tabela.min(axis=0) == 0).sum()), "de", tabela.shape[1])
print("exemplos que todos erram:      ", int((tabela.max(axis=0) == 0).sum()))
summarize_classification_results(res).set_index("model")[["accuracy", "balanced_accuracy", "f1"]].round(3)

*Sua análise:* 

# Para ir além

Para quem terminou antes ou quer continuar depois. Estas tarefas não contam no tempo do hands-on e usam os objetos das tarefas anteriores.

## Extra 1 · Os modelos sabem quando erram? (~4 min)

Além da classe, cada modelo dá uma probabilidade. Transformamos isso em confiança (0 = em cima do muro, 1 = certeza total) e comparamos acertos com erros.

**Pergunta.** Algum modelo erra com a mesma convicção com que acerta? Em que aplicação isso seria um problema? E o que as probabilidades dizem sobre os exemplos 200 e 248, que todos erram?

**Para explorar.** Volte ao mapa da tarefa 4 e localize os exemplos 200 e 248. A confiança dos modelos ajuda a separar rótulo trocado de exemplo ambíguo?

In [ ]:
import numpy as np

# Confiança na classe prevista: 0 = em cima do muro (P = 0,5); 1 = certeza total (P = 0 ou 1)
resultados["confianca"] = (resultados["predicted_probability"] - 0.5).abs() * 2

rng = np.random.default_rng(0)
fig, ax = plt.subplots()
for posicao, (modelo, grupo) in enumerate(resultados.groupby("model")):
    for acerto, cor, desvio, nome in [(1, "#4392D2", -0.15, "acertos"), (0, "#F04E44", 0.15, "erros")]:
        y = grupo.loc[grupo.correct == acerto, "confianca"]
        x = posicao + desvio + rng.uniform(-0.07, 0.07, len(y))
        ax.scatter(x, y, s=22 if acerto else 48, color=cor, alpha=0.6 if acerto else 0.95,
                   edgecolor="#1C2127", linewidth=0.3, label=nome if posicao == 0 else None)
ax.set_xticks(range(3), sorted(resultados.model.unique()))
ax.set_ylabel("confiança na classe prevista")
ax.set_title("Os modelos sabem quando erram?")
ax.legend(fontsize=8)
plt.show()

print("Confiança média:")
print(resultados.groupby(["model", "correct"])["confianca"].mean().unstack()
      .rename(columns={0: "quando erra", 1: "quando acerta"}).round(2).to_string())

# P(classe 1) dada por cada modelo aos dois exemplos que todos erram (o rótulo dos dois é 1)
resultados[resultados.example_id.isin([200, 248])].pivot(index="model", columns="example_id", values="predicted_probability").round(2)

*Sua análise:* 

## Extra 2 · Outro sorteio, outro vencedor? (~4 min)

A tabela da tarefa 2 veio de um único sorteio de quais 90 exemplos ficam para teste. Repetimos a avaliação com 20 sorteios diferentes dos mesmos dados.

**Pergunta.** A diferença entre os modelos na tarefa 2 sobrevive a outros sorteios? O que isso muda na forma de reportar um resultado?

**Para explorar.** Aumente para 50 sorteios, ou mude `test_size` em `evaluate_models_on_dataset` (passe o mouse sobre ela). Com um teste maior, a variação diminui?

In [ ]:
import numpy as np
import pandas as pd

# Mesmos 300 pontos, 20 sorteios diferentes de quais 90 ficam para teste
acuracias = pd.DataFrame({
    semente: evaluate_models_on_dataset(df, scenario="sorteio", random_state=semente).groupby("model")["correct"].mean()
    for semente in range(20)
}).T

rng = np.random.default_rng(0)
fig, ax = plt.subplots()
for posicao, modelo in enumerate(acuracias.columns):
    ax.scatter(posicao + rng.uniform(-0.12, 0.12, 20), acuracias[modelo], s=30, color="#4392D2", alpha=0.6, edgecolor="#1C2127", linewidth=0.3)
    ax.hlines(acuracias[modelo].mean(), posicao - 0.25, posicao + 0.25, color="#F04E44", linewidth=2.6)
ax.set_xticks(range(len(acuracias.columns)), acuracias.columns)
ax.set_ylabel("acurácia no teste")
ax.set_title("20 sorteios do conjunto de teste (traço = média)")
plt.show()

print("Quantas vezes cada modelo ficou em primeiro (empates contam para todos os empatados):")
print((acuracias.eq(acuracias.max(axis=1), axis=0)).sum().to_string())
acuracias.describe().loc[["mean", "std", "min", "max"]].round(3)

*Sua análise:* 